# Google WeatherNext 2 Forecast Demo
This notebook demonstrates how to load, slice, and plot the WeatherNext 2 forecasting dataset from Google Cloud Storage over the Philippines, matching the exact styling of your GFS MSLP, Precipitation, and Thickness maps. It also generates the full 15-day forecast sequence (60 frames) with province boundaries.

In [ ]:
# 1. Install required libraries
!pip install -q cartopy gcsfs zarr xarray matplotlib numpy scipy shapely

In [ ]:
# 2. Authenticate Google Cloud account and check for Province boundaries
import os
import gcsfs

# Determine if running inside Google Colab web environment
is_colab = False
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Google Colab!")
    fs = gcsfs.GCSFileSystem()
    is_colab = True
except ImportError:
    print("Running locally. Initiating interactive browser authentication...")
    print("Please check the cell output below for a link to open in your browser.")
    fs = gcsfs.GCSFileSystem(token='browser')

# If running in Google Colab web editor, prompt user to upload ph_provinces.json if not present
if is_colab and not os.path.exists('ph_provinces.json') and not os.path.exists('public/data/ph_provinces.json'):
    from google.colab import files
    print("\n--- Province Borders Setup ---")
    print("To render province borders, please upload the 'ph_provinces.json' file from your local workspace:")
    try:
        uploaded = files.upload()
    except Exception as e:
        print("Upload skipped or failed. Continuing without boundaries.")

In [ ]:
# 3. Find and open the Zarr group inside the latest forecast run
import xarray as xr
import gcsfs

if 'fs' not in locals():
    fs = gcsfs.GCSFileSystem()

parent_path = 'gs://weathernext/weathernext_2_0_0_mean/zarr/2025_to_present'
all_items = fs.ls(parent_path)
run_folders = [f'gs://{item}' for item in all_items if item.endswith('_preds')]

if not run_folders:
    raise RuntimeError('No forecast run folders found!')

run_folders.sort()
latest_run_path = run_folders[-1]
latest_zarr_path = f'{latest_run_path}/predictions.zarr'
print('Opening Zarr dataset at:', latest_zarr_path)

store = fs.get_mapper(latest_zarr_path)
ds = xr.open_zarr(store, consolidated=True)

print('Available variables:')
print(list(ds.data_vars))

# Determine if latitude is in increasing or decreasing order, and slice to match the GFS region bounds
LAT_MIN, LAT_MAX = 2.0, 28.0
LON_MIN, LON_MAX = 112.0, 140.0

if ds.lat[0] < ds.lat[-1]:
    ds_ph = ds.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
else:
    ds_ph = ds.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))

print('Philippines subset metadata:')
print(ds_ph)

In [ ]:
# 4. Slice for a specific forecast step and plot wind + precipitation (with Provinces)
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import scipy.ndimage
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.lines as mlines
from shapely.geometry import shape
import json

# Select target step (e.g. index 12 in the time dimension)
ds_target = ds_ph.isel(time=12)

# Load province boundaries
province_shapely_geometries = []
geojson_paths = ['public/data/ph_provinces.json', 'ph_provinces.json']
for p_path in geojson_paths:
    if os.path.exists(p_path):
        try:
            with open(p_path, 'r', encoding='utf-8') as f:
                geojson_content_dict = json.load(f)
            province_shapely_geometries = [shape(prov_feat['geometry']) for prov_feat in geojson_content_dict['features']]
            print(f"Loaded province boundaries from: {p_path}")
            break
        except Exception as e:
            print(f"Failed to load {p_path}: {e}")

precip_rate = (ds_target['total_precipitation_6hr'] * 1000.0).values
msl_data = (ds_target['mean_sea_level_pressure'] / 100.0).values
phi_500 = ds_target['geopotential'].sel(level=500)
phi_1000 = ds_target['geopotential'].sel(level=1000)
thickness = ((phi_500 - phi_1000) / 98.0665).values

X, Y = np.meshgrid(ds_ph.lon, ds_ph.lat)

fig = plt.figure(figsize=(14, 11))
fig.subplots_adjust(top=0.88)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([112.0, 140.0, 2.0, 28.0], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor='#eaeaea', zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor='#d4e5ed', zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor='#222', zorder=5)
ax.add_feature(cfeature.BORDERS, linestyle='-', linewidth=0.6, edgecolor='#555', zorder=5)

# Add Provinces
if province_shapely_geometries:
    ax.add_geometries(province_shapely_geometries, crs=ccrs.PlateCarree(),
                      facecolor='none', edgecolor='#555555', linewidth=0.4, alpha=0.6, zorder=3)

gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.4, linestyle=':', zorder=6)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10, 'color': '#333'}
gl.ylabel_style = {'size': 10, 'color': '#333'}

pr_levels = [0, 0.5, 1, 2, 5, 8, 12, 18, 25, 35, 45, 55, 70, 85, 100, 150]
pr_colors = [
    '#ffffff00', '#dbe9f6', '#a6cbe3', '#5ba3d0', '#227abb',
    '#4ac15e', '#2ea946', '#1a862f', '#ffdb00', '#f7a800',
    '#ea7200', '#df4000', '#d41c00', '#b40047', '#c432b4'
]
pr_cmap = ListedColormap(pr_colors)
pr_cmap.set_over('#4b0082')
pr_norm = BoundaryNorm(pr_levels, ncolors=len(pr_colors), clip=False)

cf = ax.contourf(
    X, Y, precip_rate, levels=pr_levels, cmap=pr_cmap, norm=pr_norm,
    extend='max', transform=ccrs.PlateCarree(), zorder=2
)
cb = fig.colorbar(cf, ax=ax, orientation='vertical', pad=0.02, shrink=0.85, aspect=25)
cb.set_ticks(pr_levels)
cb.ax.tick_params(labelsize=9)
cb.set_label('6-hr Precipitation (mm)', fontsize=10)
cb.outline.set_edgecolor('black')
cb.outline.set_linewidth(1)

msl_smooth = scipy.ndimage.gaussian_filter(msl_data, sigma=1)
cs = ax.contour(
    X, Y, msl_smooth, levels=range(900, 1050, 4),
    colors='black', linewidths=1.2, transform=ccrs.PlateCarree(), zorder=3
)
ax.clabel(cs, inline=True, fontsize=9, fmt='%d', colors='black')

thick_smooth = scipy.ndimage.gaussian_filter(thickness, sigma=1.5)
thick_levels = list(range(492, 600, 6))
ct = ax.contour(
    X, Y, thick_smooth, levels=thick_levels,
    colors='#2563eb', linewidths=0.8, linestyles='dashed',
    transform=ccrs.PlateCarree(), zorder=3
)
ax.clabel(ct, inline=True, fontsize=8, fmt='%d', colors='#2563eb')

ct540 = ax.contour(
    X, Y, thick_smooth, levels=[540],
    colors='#dc2626', linewidths=2.5, linestyles='solid',
    transform=ccrs.PlateCarree(), zorder=4
)
ax.clabel(ct540, inline=True, fontsize=10, fmt='%d', colors='#dc2626')

PAR_LONS = [115.0, 115.0, 120.0, 120.0, 135.0, 135.0, 115.0]
PAR_LATS = [5.0, 15.0, 21.0, 25.0, 25.0, 5.0, 5.0]
ax.plot(PAR_LONS, PAR_LATS, transform=ccrs.PlateCarree(),
        color='#d62728', linestyle='-', linewidth=2.5, zorder=7)

init_time = str(ds_target.init_time.values)[:16]
forecast_time = str(ds_target.datetime.values)[:16]
lead_hours = int(ds_target.time.values / np.timedelta64(1, 'h'))
fh_str = f'f{lead_hours:03d}'

pos = ax.get_position()
left, right = pos.x0, pos.x1
y_top = pos.y1 + 0.045
y_bottom = pos.y1 + 0.015
y_line = pos.y1 + 0.005

fig.text(left, y_top, 'Philippine T/W', ha='left', va='bottom', fontsize=14, weight='bold', color='#888')
fig.text(right, y_top, '6-hr Precip (mm), MSLP (hPa) & 1000-500 mb Thickness (dam)',
         ha='right', va='bottom', fontsize=12, weight='bold', color='black')
fig.text(left, y_bottom, f'Model: WeatherNext 2 | Forecast Hour: {fh_str}',
         ha='left', va='bottom', fontsize=11, color='black')
fig.text(right, y_bottom, f'Init: {init_time} / Valid: {forecast_time}',
         ha='right', va='bottom', fontsize=11, color='black')

sep = mlines.Line2D((left, right), (y_line, y_line), color='black', linewidth=1, transform=fig.transFigure)
fig.add_artist(sep)

plt.show()

In [ ]:
# 5. Save all 15-day forecast frames (60 frames) as PNGs to drive or local folder
import os
import scipy.ndimage

frames_dir = 'weathernext_frames'
os.makedirs(frames_dir, exist_ok=True)
total_steps = len(ds_ph.time)

print(f"Generating 15-day forecast frames (60 steps) and saving to '{frames_dir}/'...")

for i in range(total_steps):
    ds_target = ds_ph.isel(time=i)
    lead_hours = int(ds_target.time.values / np.timedelta64(1, 'h'))
    init_time = str(ds_target.init_time.values)[:16]
    forecast_time = str(ds_target.datetime.values)[:16]
    fh_str = f'f{lead_hours:03d}'
    
    # Load variables
    precip_rate = (ds_target['total_precipitation_6hr'] * 1000.0).values
    msl_data = (ds_target['mean_sea_level_pressure'] / 100.0).values
    phi_500 = ds_target['geopotential'].sel(level=500)
    phi_1000 = ds_target['geopotential'].sel(level=1000)
    thickness = ((phi_500 - phi_1000) / 98.0665).values
    
    fig = plt.figure(figsize=(14, 11))
    fig.subplots_adjust(top=0.88)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([112.0, 140.0, 2.0, 28.0], crs=ccrs.PlateCarree())
    
    ax.add_feature(cfeature.LAND, facecolor='#eaeaea', zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor='#d4e5ed', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor='#222', zorder=5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', linewidth=0.6, edgecolor='#555', zorder=5)
    
    if province_shapely_geometries:
        ax.add_geometries(province_shapely_geometries, crs=ccrs.PlateCarree(),
                          facecolor='none', edgecolor='#555555', linewidth=0.4, alpha=0.6, zorder=3)
                          
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.4, linestyle=':', zorder=6)
    gl.top_labels = False
    gl.right_labels = False
    
    cf = ax.contourf(
        X, Y, precip_rate, levels=pr_levels, cmap=pr_cmap, norm=pr_norm,
        extend='max', transform=ccrs.PlateCarree(), zorder=2
    )
    cb = fig.colorbar(cf, ax=ax, orientation='vertical', pad=0.02, shrink=0.85, aspect=25)
    cb.set_ticks(pr_levels)
    cb.set_label('6-hr Precipitation (mm)', fontsize=10)
    cb.outline.set_edgecolor('black')
    
    msl_smooth = scipy.ndimage.gaussian_filter(msl_data, sigma=1)
    cs = ax.contour(
        X, Y, msl_smooth, levels=range(900, 1050, 4),
        colors='black', linewidths=1.2, transform=ccrs.PlateCarree(), zorder=3
)
    ax.clabel(cs, inline=True, fontsize=9, fmt='%d', colors='black')
    
    thick_smooth = scipy.ndimage.gaussian_filter(thickness, sigma=1.5)
    ct = ax.contour(
        X, Y, thick_smooth, levels=thick_levels,
        colors='#2563eb', linewidths=0.8, linestyles='dashed',
        transform=ccrs.PlateCarree(), zorder=3
)
    ax.clabel(ct, inline=True, fontsize=8, fmt='%d', colors='#2563eb')
    
    ct540 = ax.contour(
        X, Y, thick_smooth, levels=[540],
        colors='#dc2626', linewidths=2.5, linestyles='solid',
        transform=ccrs.PlateCarree(), zorder=4
)
    ax.clabel(ct540, inline=True, fontsize=10, fmt='%d', colors='#dc2626')
    
    ax.plot(PAR_LONS, PAR_LATS, transform=ccrs.PlateCarree(),
            color='#d62728', linestyle='-', linewidth=2.5, zorder=7)
            
    pos = ax.get_position()
    left, right = pos.x0, pos.x1
    y_top = pos.y1 + 0.045
    y_bottom = pos.y1 + 0.015
    y_line = pos.y1 + 0.005
    
    fig.text(left, y_top, 'Philippine T/W', ha='left', va='bottom', fontsize=14, weight='bold', color='#888')
    fig.text(right, y_top, '6-hr Precip (mm), MSLP (hPa) & 1000-500 mb Thickness (dam)',
             ha='right', va='bottom', fontsize=12, weight='bold', color='black')
    fig.text(left, y_bottom, f'Model: WeatherNext 2 | Forecast Hour: {fh_str}',
             ha='left', va='bottom', fontsize=11, color='black')
    fig.text(right, y_bottom, f'Init: {init_time} / Valid: {forecast_time}',
             ha='right', va='bottom', fontsize=11, color='black')
             
    sep = mlines.Line2D((left, right), (y_line, y_line), color='black', linewidth=1, transform=fig.transFigure)
    fig.add_artist(sep)
    
    plt.savefig(f"{frames_dir}/frame_{i:02d}.png", dpi=100, bbox_inches='tight', facecolor='white')
    plt.close()
    
    if (i + 1) % 10 == 0 or (i + 1) == total_steps:
        print(f"  Progress: {i+1}/{total_steps} frames saved.")

print(f"\nAll {total_steps} frames successfully generated and saved to the '{frames_dir}/' folder!")